In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Zdalna transpilacja Circuit za pomocą Qiskit Transpiler Service

> **Danger:** Od 18 lipca 2025 r. usługa jest migrowana do obsługi nowej platformy IBM Quantum&reg; i jest niedostępna. W przypadku przejść AI możesz skorzystać z [trybu lokalnego](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Usługa jest wydaniem beta i może ulec zmianie.
>     Jeśli masz uwagi lub chcesz skontaktować się z zespołem programistów, użyj tego kanału [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Qiskit Transpiler Service oferuje możliwości transpilacji w chmurze. Oprócz lokalnych możliwości Transpilatora Qiskit, twoje zadania transpilacji mogą korzystać zarówno z zasobów chmurowych IBM Quantum, jak i przejść Transpilatora opartych na AI.

Qiskit Transpiler Service udostępnia bibliotekę Pythona umożliwiającą płynną integrację tej usługi i jej możliwości z bieżącymi wzorcami i przepływami pracy Qiskit. Usługa jest dostępna wyłącznie dla użytkowników planów IBM Quantum Premium Plan, Flex Plan i On-Prem (przez API IBM Quantum Platform).

<span id="install-transpiler-service"></span>
## Instalacja pakietu qiskit-ibm-transpiler
Aby korzystać z Qiskit Transpiler Service, zainstaluj pakiet `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Pakiet automatycznie uwierzytelnia się przy użyciu [poświadczeń IBM Quantum Platform](/guides/cloud-setup) zgodnie ze sposobem, w jaki [Qiskit Runtime nimi zarządza](/guides/initialize-account):
- Zmienna środowiskowa: `QISKIT_IBM_TOKEN`
- Plik konfiguracyjny `~/.qiskit/qiskit-ibm.json` (w sekcji `default-ibm-quantum`).

*Uwaga*: Ten pakiet wymaga Qiskit SDK v1.X.

## Opcje transpilacji qiskit-ibm-transpiler
- `backend_name` (opcjonalne, str) – nazwa Backend zgodna z oczekiwaniami QiskitRuntimeService (na przykład `ibm_torino`). Jeśli jest ustawiona, metoda transpilacji używa układu z podanego Backend do operacji transpilacji. Jeśli ustawiona jest inna opcja wpływająca na te ustawienia, np. `coupling_map`, ustawienia `backend_name` są nadpisywane.
- `coupling_map` (opcjonalne, List[List[int]]) – prawidłowa lista mapy sprzężeń (na przykład [[0,1],[1,2]]). Jeśli jest ustawiona, metoda transpilacji używa tej mapy sprzężeń do operacji transpilacji. Jeśli jest zdefiniowana, nadpisuje każdą wartość podaną dla `target`.
- `optimization_level` (int) – potencjalny poziom optymalizacji stosowany podczas procesu transpilacji. Prawidłowe wartości to [1,2,3], gdzie 1 oznacza najmniejszą optymalizację (i najszybsze działanie), a 3 – największą optymalizację (i największą czasochłonność).
- `ai` ("true", "false", "auto") – czy podczas transpilacji mają być używane możliwości oparte na AI. Dostępne możliwości AI mogą obejmować przejścia transpilacji `AIRouting` lub inne metody syntezy oparte na AI. Jeśli ta wartość to `"true"`, usługa stosuje różne przejścia transpilacji oparte na AI w zależności od żądanego `optimization_level`. Jeśli `"false"`, używane są najnowsze funkcje transpilacji Qiskit bez AI. Jeśli `"auto"`, usługa decyduje, czy zastosować standardowe heurystyczne przejścia Qiskit, czy przejścia oparte na AI na podstawie twojego Circuit.
- `qiskit_transpile_options` (dict) – obiekt słownika Pythona, który może zawierać dowolną inną opcję prawidłową dla [metody `transpile()` w Qiskit](defaults-and-configuration-options). Jeśli wejście `qiskit_transpile_options` zawiera `optimization_level`, jest on odrzucany na rzecz wartości `optimization_level` podanej jako parametr wejściowy. Jeśli `qiskit_transpile_options` zawiera opcję nierozpoznaną przez metodę `transpile()` Qiskit, biblioteka zgłasza błąd.

Więcej informacji o dostępnych metodach `qiskit-ibm-transpiler` znajdziesz w [dokumentacji API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Aby dowiedzieć się więcej o API usługi, zapoznaj się z [dokumentacją REST API Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Przykłady
Poniższe przykłady pokazują, jak transpilować Circuit za pomocą Qiskit Transpiler Service z różnymi parametrami.

1. Utwórz Circuit i wywołaj Qiskit Transpiler Service, aby przetranspilować go z `ibm_torino` jako `backend_name`, 3 jako `optimization_level` i bez użycia AI podczas transpilacji.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Uwaga*: jako `backend_name` możesz używać tylko urządzeń Backend, do których masz dostęp ze swoim kontem IBM Quantum. Oprócz `backend_name`, `TranspilerService` akceptuje również `coupling_map` jako parametr.

2. Utwórz podobny Circuit i przetranspiluj go, żądając możliwości transpilacji AI poprzez ustawienie flagi `ai` na `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Utwórz podobny Circuit i przetranspiluj go, pozwalając usłudze zdecydować, czy użyć przejść transpilacji opartych na AI.